In [1]:
# ================================
# CORE LIBRARIES
# ================================
import numpy as np
import pandas as pd

# ================================
# SCIKIT-LEARN MODELS
# ================================
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor
)

# ================================
# PREPROCESSING
# ================================
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# ================================
# MODEL SELECTION
# ================================
from sklearn.model_selection import TimeSeriesSplit

# ================================
# METRICS
# ================================
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# ================================
# STATISTICAL TESTING
# ================================
from scipy import stats

# ================================
# UTILITIES
# ================================
from itertools import combinations

from main_pipeline import (
    load_data,
    temporal_split,
    get_feature_groups,
    build_pipeline
)


# Load dataset
df = load_data("master_dataset_enhanced.csv")

train, test = temporal_split(df)

X_train = train.drop(columns=["yield"])
X_test = test.drop(columns=["yield"])
y_train = train["yield"]
y_test = test["yield"]

# Get feature groups
group1, group2, group3, group4, group5 = get_feature_groups()

# Define full feature set
categorical_cols = ['crop', 'state', 'season']

full_features = (
    group1 +
    group2 +
    group3 +
    group4 +
    group5 +
    categorical_cols
)

In [3]:
print("\n" + "="*70)
print("📊 TASK 3.1: MODEL COMPARISON STATISTICAL TEST")
print("="*70)

models_to_compare = {
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "Extra Trees": ExtraTreesRegressor(n_estimators=200, random_state=42)
}

predictions = {}

# ---------------------------------------
# Proper pipeline builder (WITH encoding)
# ---------------------------------------

def build_pipeline_with_model(feature_list, model):

    X_sample = X_train[feature_list]

    num_cols = X_sample.select_dtypes(include=['int64','float64']).columns.tolist()
    cat_cols = X_sample.select_dtypes(include=['object']).columns.tolist()

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, num_cols),
        ('cat', categorical_pipeline, cat_cols)
    ])

    return Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])


# ---------------------------------------
# Train models correctly
# ---------------------------------------

for name, model in models_to_compare.items():

    pipe = build_pipeline_with_model(full_features, model)

    pipe.fit(X_train[full_features], y_train)

    predictions[name] = pipe.predict(X_test[full_features])


# -----------------------------------------
# Bootstrap comparison on R²
# -----------------------------------------

n_bootstrap = 1000
np.random.seed(42)

rf_scores = []
gb_scores = []

rf_pred = predictions["Random Forest"]
gb_pred = predictions["Gradient Boosting"]

for _ in range(n_bootstrap):
    indices = np.random.choice(len(y_test), len(y_test), replace=True)

    rf_scores.append(r2_score(y_test.iloc[indices], rf_pred[indices]))
    gb_scores.append(r2_score(y_test.iloc[indices], gb_pred[indices]))

rf_scores = np.array(rf_scores)
gb_scores = np.array(gb_scores)

# Paired t-test on R² distributions
t_stat, p_value = stats.ttest_rel(rf_scores, gb_scores)

mean_diff = np.mean(rf_scores - gb_scores)
ci_low = np.percentile(rf_scores - gb_scores, 2.5)
ci_high = np.percentile(rf_scores - gb_scores, 97.5)

print("\nRF vs GB")
print(f"Mean ΔR²: {mean_diff:.6f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"95% CI of ΔR²: [{ci_low:.6f}, {ci_high:.6f}]")


📊 TASK 3.1: MODEL COMPARISON STATISTICAL TEST

RF vs GB
Mean ΔR²: 0.033716
t-statistic: 169.3369
p-value: 0.000000
95% CI of ΔR²: [0.022333, 0.046880]


In [5]:
print("\n" + "="*70)
print("📊 TASK 3.2: 10-FOLD STABILITY ANALYSIS")
print("="*70)

# Ensure chronological order
train_sorted = train.sort_values("year")

X_train_sorted = train_sorted.drop(columns=["yield"])
y_train_sorted = train_sorted["yield"]

tscv = TimeSeriesSplit(n_splits=10)

r2_scores = []
mae_scores = []
rmse_scores = []

# ---------------------------------------
# Proper pipeline builder (WITH encoding)
# ---------------------------------------

def build_pipeline_with_model(feature_list, model):

    X_sample = X_train_sorted[feature_list]

    num_cols = X_sample.select_dtypes(include=['int64','float64']).columns.tolist()
    cat_cols = X_sample.select_dtypes(include=['object']).columns.tolist()

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, num_cols),
        ('cat', categorical_pipeline, cat_cols)
    ])

    return Pipeline([
        ('preprocessing', preprocessor),
        ('model', ExtraTreesRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ))
    ])

# ---------------------------------------
# TimeSeries CV
# ---------------------------------------

for train_idx, val_idx in tscv.split(X_train_sorted):

    X_tr = X_train_sorted.iloc[train_idx]
    X_val = X_train_sorted.iloc[val_idx]

    y_tr = y_train_sorted.iloc[train_idx]
    y_val = y_train_sorted.iloc[val_idx]

    model = build_pipeline_with_model(full_features, ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))

    model.fit(X_tr[full_features], y_tr)

    preds = model.predict(X_val[full_features])

    r2_scores.append(r2_score(y_val, preds))
    mae_scores.append(mean_absolute_error(y_val, preds))
    rmse_scores.append(np.sqrt(mean_squared_error(y_val, preds)))

print(f"R²: {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")
print(f"MAE: {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"RMSE: {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")


📊 TASK 3.2: 10-FOLD STABILITY ANALYSIS
R²: 0.9318 ± 0.0335
MAE: 0.9065 ± 0.1446
RMSE: 3.6731 ± 0.9135


In [7]:
print("\n" + "="*70)
print("📊 TASK 3.3: BOOTSTRAP RELIABILITY")
print("="*70)

# ---------------------------------------
# Proper pipeline builder (WITH encoding)
# ---------------------------------------

def build_pipeline_with_model(feature_list, model):

    X_sample = X_train[feature_list]

    num_cols = X_sample.select_dtypes(include=['int64','float64']).columns.tolist()
    cat_cols = X_sample.select_dtypes(include=['object']).columns.tolist()

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, num_cols),
        ('cat', categorical_pipeline, cat_cols)
    ])

    return Pipeline([
        ('preprocessing', preprocessor),
        ('model', ExtraTreesRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ))
    ])


# ---------------------------------------
# Train final model correctly
# ---------------------------------------

final_model = build_pipeline_with_model(
    full_features,
    ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
)

final_model.fit(X_train[full_features], y_train)

final_pred = final_model.predict(X_test[full_features])


# ---------------------------------------
# Bootstrap reliability
# ---------------------------------------

n_boot = 1000
r2_boot = []

np.random.seed(42)

for _ in range(n_boot):
    idx = np.random.choice(len(y_test), len(y_test), replace=True)
    r2_boot.append(r2_score(y_test.iloc[idx], final_pred[idx]))

r2_boot = np.array(r2_boot)

ci_low = np.percentile(r2_boot, 2.5)
ci_high = np.percentile(r2_boot, 97.5)

print(f"Bootstrap Mean R²: {np.mean(r2_boot):.4f}")
print(f"R² 95% Prediction Interval: [{ci_low:.4f}, {ci_high:.4f}]")


📊 TASK 3.3: BOOTSTRAP RELIABILITY
Bootstrap Mean R²: 0.9294
R² 95% Prediction Interval: [0.8905, 0.9608]


In [8]:
results_df = pd.DataFrame({
    "R2_CV_Mean": [np.mean(r2_scores)],
    "R2_CV_Std": [np.std(r2_scores)],
    "MAE_CV_Mean": [np.mean(mae_scores)],
    "MAE_CV_Std": [np.std(mae_scores)],
    "RMSE_CV_Mean": [np.mean(rmse_scores)],
    "RMSE_CV_Std": [np.std(rmse_scores)],
    "Bootstrap_R2_Lower": [ci_low],
    "Bootstrap_R2_Upper": [ci_high],
    "RF_vs_GB_pvalue": [p_value]
})

results_df.to_excel("statistical_tests.xlsx", index=False)

print("✓ Saved: statistical_tests.xlsx")

✓ Saved: statistical_tests.xlsx


In [9]:
print("\n" + "="*70)
print("📊 CORRECTED MODEL COMPARISON")
print("="*70)

from itertools import combinations
from scipy import stats

def build_pipeline_with_model(feature_list, model):
    
    X_train_sub = X_train[feature_list]
    
    num_cols = X_train_sub.select_dtypes(include=['int64','float64']).columns.tolist()
    cat_cols = X_train_sub.select_dtypes(include=['object']).columns.tolist()
    
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])
    
    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, num_cols),
        ('cat', categorical_pipeline, cat_cols)
    ])
    
    return Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

model_defs = {
    "Extra Trees": ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, random_state=42)
}

predictions = {}

for name, model in model_defs.items():
    pipe = build_pipeline_with_model(full_features, model)
    pipe.fit(X_train[full_features], y_train)
    predictions[name] = pipe.predict(X_test[full_features])

comparison_results = []

for model_a, model_b in combinations(model_defs.keys(), 2):
    
    pred_a = predictions[model_a]
    pred_b = predictions[model_b]
    
    # Bootstrap difference in R²
    # Bootstrap difference in R²
    n_boot = 1000
    diffs = []

    np.random.seed(42)

    for _ in range(n_boot):
        idx = np.random.choice(len(y_test), len(y_test), replace=True)

        r2_a = r2_score(y_test.iloc[idx], pred_a[idx])
        r2_b = r2_score(y_test.iloc[idx], pred_b[idx])

        diffs.append(r2_a - r2_b)

    diffs = np.array(diffs)

    mean_diff = np.mean(diffs)
    ci_low = np.percentile(diffs, 2.5)
    ci_high = np.percentile(diffs, 97.5)

    # Correct statistical test
    t_stat, p_value = stats.ttest_1samp(diffs, 0)
        
    comparison_results.append({
        "Model_A": model_a,
        "Model_B": model_b,
        "Mean_R2_Diff": mean_diff,
        "95%_CI_Lower": ci_low,
        "95%_CI_Upper": ci_high,
        "p_value": p_value
    })

comparison_df = pd.DataFrame(comparison_results)
print(comparison_df)



📊 CORRECTED MODEL COMPARISON
         Model_A            Model_B  Mean_R2_Diff  95%_CI_Lower  95%_CI_Upper  \
0    Extra Trees      Random Forest      0.002644     -0.014926      0.018728   
1    Extra Trees  Gradient Boosting      0.036359      0.012891      0.059747   
2  Random Forest  Gradient Boosting      0.033716      0.022333      0.046880   

        p_value  
0  8.173755e-22  
1  0.000000e+00  
2  0.000000e+00  


In [11]:
print("\n" + "="*70)
print("📊 10-FOLD STABILITY (WITH RMSE)")
print("="*70)

# Ensure chronological order
train_sorted = train.sort_values("year")

X_train_sorted = train_sorted.drop(columns=["yield"])
y_train_sorted = train_sorted["yield"]

tscv = TimeSeriesSplit(n_splits=10)

r2_scores = []
mae_scores = []
rmse_scores = []

# ---------------------------------------
# Proper pipeline builder (WITH encoding)
# ---------------------------------------

def build_pipeline_with_model(feature_list, model):

    X_sample = X_train_sorted[feature_list]

    num_cols = X_sample.select_dtypes(include=['int64','float64']).columns.tolist()
    cat_cols = X_sample.select_dtypes(include=['object']).columns.tolist()

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, num_cols),
        ('cat', categorical_pipeline, cat_cols)
    ])

    return Pipeline([
        ('preprocessing', preprocessor),
        ('model', ExtraTreesRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ))
    ])

# ---------------------------------------
# TimeSeries Cross-Validation
# ---------------------------------------

for train_idx, val_idx in tscv.split(X_train_sorted):

    X_tr = X_train_sorted.iloc[train_idx]
    X_val = X_train_sorted.iloc[val_idx]

    y_tr = y_train_sorted.iloc[train_idx]
    y_val = y_train_sorted.iloc[val_idx]

    model = build_pipeline_with_model(
        full_features,
        ExtraTreesRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    )

    model.fit(X_tr[full_features], y_tr)

    preds = model.predict(X_val[full_features])

    r2_scores.append(r2_score(y_val, preds))
    mae_scores.append(mean_absolute_error(y_val, preds))
    rmse_scores.append(np.sqrt(mean_squared_error(y_val, preds)))

print(f"R²: {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")
print(f"MAE: {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"RMSE: {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")


📊 10-FOLD STABILITY (WITH RMSE)
R²: 0.9318 ± 0.0335
MAE: 0.9065 ± 0.1446
RMSE: 3.6731 ± 0.9135


In [12]:
"""You already computed:

R² 95% Prediction Interval: [0.8885, 0.9608]

Just document:

Bootstrap iterations: 1000
"""

'You already computed:\n\nR² 95% Prediction Interval: [0.8885, 0.9608]\n\nJust document:\n\nBootstrap iterations: 1000\n'

In [13]:
with pd.ExcelWriter("statistical_tests.xlsx") as writer:
    
    # Model Comparisons
    comparison_df.round(6).to_excel(
        writer,
        sheet_name="Model_Comparisons",
        index=False
    )
    
    # Cross-Validation Stability
    pd.DataFrame({
        "R2_Mean": [np.mean(r2_scores)],
        "R2_Std": [np.std(r2_scores)],
        "MAE_Mean": [np.mean(mae_scores)],
        "MAE_Std": [np.std(mae_scores)],
        "RMSE_Mean": [np.mean(rmse_scores)],
        "RMSE_Std": [np.std(rmse_scores)]
    }).round(6).to_excel(
        writer,
        sheet_name="CV_Stability",
        index=False
    )
    
    # Bootstrap Reliability
    pd.DataFrame({
        "Bootstrap_R2_Mean": [np.mean(r2_boot)],
        "Bootstrap_R2_Lower": [ci_low],
        "Bootstrap_R2_Upper": [ci_high]
    }).round(6).to_excel(
        writer,
        sheet_name="Bootstrap_Reliability",
        index=False
    )

print("✓ statistical_tests.xlsx saved successfully")


✓ statistical_tests.xlsx saved successfully


In [14]:
print("Fold-wise RMSE:", rmse_scores)

Fold-wise RMSE: [np.float64(3.6669959530780294), np.float64(2.4460238711797997), np.float64(2.527171285407081), np.float64(3.037935979692478), np.float64(4.967037365207313), np.float64(4.9943645762018765), np.float64(4.596770187598867), np.float64(2.865636989049114), np.float64(3.5349260550336825), np.float64(4.094410614061203)]
